# Build a custom vector database operator

This guide onboards a vector database to NeMo Retriever Library (NRL) with a runnable in-memory DuckDB example. It uses exhaustive cosine-distance search to explain the contract. Replace it with your database's native vector index for production.


## Before you begin

Install DuckDB with `pip install 'nemo-retriever[tabular]'`. A custom `VDB` receives nested batches of canonical embedded records through `run(records)` and `write_to_index(records)`. Its `retrieval(vectors)` method receives precomputed query vectors, not query strings. Current NRL also requires collection and document API methods. This fixed-table tutorial implements their method surface and raises an explicit unsupported-operation error; implement durable collection behavior for a service collection backend.


## Step 1: Implement the fixed-table API

Copy the class into an importable module, then replace DuckDB connection, schema, and Python ranking with your backend client. Preserve the validation and vector-first retrieval boundary.


In [ ]:
from math import sqrt
from typing import Any
from uuid import uuid4
import duckdb
from nemo_retriever.common.schemas.collections import (CollectionCreateRequest, CollectionDeleteResult, CollectionInfo, CollectionPage, CollectionUpdateRequest, DocumentDeleteResult, DocumentInfo, DocumentPage)
from nemo_retriever.common.vdb.adt_vdb import CollectionWriteContext, CollectionWriteResult, UnsupportedVDBOperation, VDB, VDBInvalidRequest

class DuckDBVDB(VDB):
    """In-memory teaching backend; not a production ANN backend."""
    def __init__(self, *, vector_dim: int, database: str = ':memory:', **kwargs: Any):
        super().__init__(vector_dim=vector_dim, database=database, **kwargs)
        self.vector_dim, self.db = vector_dim, duckdb.connect(database)
        self.create_index()
    def create_index(self, **kwargs: Any):
        if kwargs.get('recreate'): self.db.execute('DROP TABLE IF EXISTS nrl_vectors')
        self.db.execute('CREATE TABLE IF NOT EXISTS nrl_vectors (chunk_id VARCHAR PRIMARY KEY, vector FLOAT[], text VARCHAR, metadata JSON)')
    def write_to_index(self, records: list, **kwargs: Any):
        rows = []
        for batch in records:
            for record in batch:
                metadata = dict(record.get('metadata') or {})
                vector, text = metadata.get('embedding'), record.get('text') or metadata.get('content')
                if not isinstance(vector, (list, tuple)) or len(vector) != self.vector_dim: raise VDBInvalidRequest(f'Expected an embedding with {self.vector_dim} dimensions')
                if not isinstance(text, str) or not text.strip(): raise VDBInvalidRequest('Each dense record requires nonblank text')
                rows.append((str(record.get('chunk_id') or uuid4()), [float(x) for x in vector], text, metadata))
        self.db.executemany('INSERT OR REPLACE INTO nrl_vectors VALUES (?, ?, ?, ?)', rows)
    def run(self, records: list):
        self.create_index(); self.write_to_index(records)
    def retrieval(self, queries: list, **kwargs: Any):
        top_k, stored, results = int(kwargs.get('top_k', 10)), self.db.execute('SELECT chunk_id, vector, text, metadata FROM nrl_vectors').fetchall(), []
        for query in queries:
            if not isinstance(query, (list, tuple)) or len(query) != self.vector_dim: raise VDBInvalidRequest(f'Expected a query vector with {self.vector_dim} dimensions')
            def distance(vector):
                denominator = sqrt(sum(x*x for x in query)) * sqrt(sum(x*x for x in vector))
                if denominator == 0: raise VDBInvalidRequest('Vectors must have nonzero magnitude')
                return 1.0 - sum(x*y for x, y in zip(query, vector)) / denominator
            results.append(sorted(({'chunk_id': row[0], 'text': row[2], 'metadata': row[3], '_distance': distance(list(row[1]))} for row in stored), key=lambda hit: hit['_distance'])[:top_k])
        return results
    @staticmethod
    def _collections_not_supported(*args, **kwargs):
        raise UnsupportedVDBOperation('Implement collection and document storage before using this backend with the service collection API.')
    create_collection = get_collection = list_collections = update_collection = delete_collection = _collections_not_supported
    get_document = list_documents = delete_document = write_collection = retrieve_collection = _collections_not_supported


## Step 2: Run a smoke test

Run this after every backend change. It proves that the class is concrete, accepts canonical record batches, and produces one ranked result list per query vector.


In [ ]:
vdb = DuckDBVDB(vector_dim=3)
records = [[
    {'chunk_id': 'intro', 'text': 'NeMo Retriever extracts and indexes content.', 'metadata': {'embedding': [1.0, 0.0, 0.0]}},
    {'chunk_id': 'duckdb', 'text': 'DuckDB stores this tutorial in memory.', 'metadata': {'embedding': [0.0, 1.0, 0.0]}},
]]
vdb.run(records)
hits = vdb.retrieval([[0.9, 0.1, 0.0]], top_k=1)
assert hits[0][0]['chunk_id'] == 'intro'
hits


## Step 3: Onboard your production backend

1. Replace `create_index()` with your backend connection, schema, and native index setup.
2. Keep the nested record-batch shape in `write_to_index()` and validate dimensions before bulk writes.
3. Keep `retrieval()` vector-first. Return one hit list per input vector, omit stored vectors, and return native distance as `_distance` when available.
4. For service collections, implement `create_collection`, collection and document CRUD, `write_collection(records, context=...)`, and `retrieve_collection(vectors, scope=..., collection_name=..., query_texts=..., top_k=...)`. Scoped retrieval returns `(hits_by_query, strategies)`.
5. Add tests for invalid dimensions, empty writes, ranking order, collection lifecycle, and concurrency.
6. Register the importable backend in `nemo_retriever.common.vdb.factory` if callers should select it by name.

The root `retriever ingest` and `retriever query` commands are LanceDB-only. Use a custom backend through the SDK or graph operators after registration.
